# 03 — Preprocessing
Clean, encode, scale and split the data.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib, os

df = pd.read_csv('../data/processed/listings_eda.csv')
print(df.shape)
print(df.columns.tolist())
print(df.shape)

(2592, 12)
['price', 'accommodates', 'bedrooms', 'bathrooms', 'neighbourhood_cleansed', 'room_type', 'minimum_nights', 'latitude', 'longitude', 'availability_365', 'number_of_reviews', 'reviews_per_month']
(2592, 12)


In [2]:
# Keep only useful features
FEATURES = ['accommodates', 'bedrooms', 'bathrooms',
            'neighbourhood_cleansed', 'room_type',
            'minimum_nights', 'latitude', 'longitude',
            'availability_365', 'number_of_reviews',
            'reviews_per_month']
TARGET = 'price'

df = df[FEATURES + [TARGET]].copy()
df = df.dropna(subset=[TARGET])
df = df[df[TARGET] > 0]
print(f'After price filter: {df.shape}')

After price filter: (2592, 12)


## Event venue proximity features
Distance (km) from each listing to major Zurich event venues.
These vary per listing via lat/lng and act as a proxy for event-driven demand.

In [3]:
# Haversine distance (km) — no external library needed
def haversine_km(lat, lon, vlat, vlon):
    R = 6371.0
    lat, lon, vlat, vlon = map(np.radians, [lat, lon, vlat, vlon])
    a = np.sin((vlat-lat)/2)**2 + np.cos(lat)*np.cos(vlat)*np.sin((vlon-lon)/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

# Major Zurich event venues (lat, lon)
VENUES = {
    "dist_hallenstadion_km": (47.4317, 8.5495),   # indoor arena
    "dist_letzigrund_km":    (47.3794, 8.5033),   # football stadium
    "dist_messe_km":         (47.4100, 8.5486),   # exhibition centre
    "dist_opernhaus_km":     (47.3654, 8.5464),   # opera house
    "dist_hb_km":            (47.3779, 8.5403),   # main station / event hub
}

for col, (vlat, vlon) in VENUES.items():
    df[col] = haversine_km(df["latitude"].values, df["longitude"].values, vlat, vlon)

df["min_dist_venue_km"] = df[[c for c in VENUES]].min(axis=1)
print("Venue distance features added:")
print(df[[*VENUES, "min_dist_venue_km"]].describe().round(2))

Venue distance features added:
       dist_hallenstadion_km  dist_letzigrund_km  dist_messe_km  \
count                2592.00             2592.00        2592.00   
mean                    6.28                3.11           4.20   
std                     2.01                1.58           1.73   
min                     0.47                0.11           0.11   
25%                     5.35                1.89           3.19   
50%                     6.60                2.98           4.47   
75%                     7.65                4.40           5.38   
max                    12.24                8.16           9.88   

       dist_opernhaus_km  dist_hb_km  min_dist_venue_km  
count            2592.00     2592.00            2592.00  
mean                2.76        2.39               1.30  
std                 1.66        1.33               0.82  
min                 0.18        0.08               0.08  
25%                 1.48        1.35               0.71  
50%              

In [4]:
# Fill missing numerics
for col in ['bedrooms', 'bathrooms', 'reviews_per_month']:
    df[col] = df[col].fillna(df[col].median())

# One-hot encode categoricals
df = pd.get_dummies(df, columns=['neighbourhood_cleansed', 'room_type'], drop_first=True)
print(f'After encoding: {df.shape}')
df.head()

After encoding: (2592, 52)


,accommodates,bedrooms,bathrooms,minimum_nights,latitude,longitude,availability_365,number_of_reviews,reviews_per_month,price,...,neighbourhood_cleansed_Sihlfeld,neighbourhood_cleansed_Unterstrass,neighbourhood_cleansed_Weinegg,neighbourhood_cleansed_Werd,neighbourhood_cleansed_Wipkingen,neighbourhood_cleansed_Witikon,neighbourhood_cleansed_Wollishofen,room_type_Hotel room,room_type_Private room,room_type_Shared room
0,2,1.0,1.5,3,47.35761,8.52131,269,0,0.885,232.0,...,False,False,False,False,False,False,False,False,False,False
1,1,1.0,1.0,5,47.36514,8.52615,92,9,0.050,48.0,...,False,False,False,False,False,False,False,False,True,False
2,3,2.0,1.5,4,47.38942,8.51881,280,31,0.180,501.0,...,False,False,False,False,False,False,False,False,False,False
3,4,2.0,1.0,4,47.37372,8.54452,13,383,2.300,160.0,...,False,False,False,False,False,False,False,False,False,False
4,6,3.0,2.0,7,47.36713,8.54868,28,13,0.080,360.0,...,False,False,False,False,False,False,False,False,False,False


In [5]:
X = df.drop(columns=[TARGET])
y = np.log1p(df[TARGET])  # log-transform: stabilises right-skewed price distribution

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (2073, 51), Test: (519, 51)


In [6]:
scaler = StandardScaler()
num_cols = ['accommodates', 'bedrooms', 'bathrooms',
            'minimum_nights', 'latitude', 'longitude',
            'availability_365', 'number_of_reviews', 'reviews_per_month',
            'dist_hallenstadion_km', 'dist_letzigrund_km', 'dist_messe_km',
            'dist_opernhaus_km', 'dist_hb_km', 'min_dist_venue_km']

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

os.makedirs('../models', exist_ok=True)
joblib.dump(scaler, '../models/scaler.pkl')

# Save splits
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv',   index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv',   index=False)
print('Splits saved.')

Splits saved.
